Foxf2iKO vs Ctrl differential expression in spots with high vs low neuron probability, and how selected genes relate to neuron probability.
- Input: `geo/Visium_Foxf2iKO_Ctrl.h5ad` (from 03)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import pearsonr
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/FOXF2/Visium")
adata = sc.read_h5ad(base / "geo" / "Visium_Foxf2iKO_Ctrl.h5ad")
adata.X = adata.layers["log1p_norm"]
adata.uns["log1p"] = {"base": None}

In [ ]:
adata.obs["neuron_prob"] = adata.obs["prob_Neurons"]
adata.obs["neuron_high"] = adata.obs["neuron_prob"] > 0.35552125
pd.crosstab(adata.obs["sample"], adata.obs["neuron_high"], normalize="index").round(2)

In [ ]:
results = {}
for high, label in [(True, "high"), (False, "low")]:
    sub = adata[adata.obs["neuron_high"] == high].copy()
    sc.tl.rank_genes_groups(sub, "genotype", groups=["Foxf2iKO"], reference="Ctrl", method="wilcoxon")
    df = sc.get.rank_genes_groups_df(sub, "Foxf2iKO").rename(columns={"names": "gene", "logfoldchanges": "log2FC", "scores": "zscore"})
    df.insert(2, "FC", 2 ** df["log2FC"])
    means = {c: pd.Series(np.expm1(np.asarray(sub[sub.obs["genotype"] == c].X.mean(0)).ravel()), index=sub.var_names) for c in ("Foxf2iKO", "Ctrl")}
    df["mean_ko"], df["mean_ctrl"] = df["gene"].map(means["Foxf2iKO"]), df["gene"].map(means["Ctrl"])
    results[label] = df.sort_values("pvals")

with pd.ExcelWriter(base / "dea" / "visium_Foxf2iKO-vs-Ctrl_wilcoxon_neuron-prob-high-vs-low.xlsx") as w:
    for label, df in results.items():
        df.to_excel(w, sheet_name=label, index=False)

In [ ]:
genes = ["Ftl1", "Tfrc", "Fth1", "Gpx4", "Trf", "Vdac2", "Prnp", "Acsl6", "Acsl4", "Pcbp1", "Slc7a11"]
fc = pd.DataFrame({k: df.set_index("gene")["FC"] for k, df in results.items()}).loc[genes]
fdr = pd.DataFrame({k: df.set_index("gene")["pvals_adj"] for k, df in results.items()}).loc[genes]
plt.figure(figsize=(2, 5))
sns.heatmap(fc, center=1, vmin=0.75, vmax=1.3, cmap="RdBu_r", annot=fdr.map(lambda p: f"{p:.1e}"), fmt="", annot_kws={"size": 6}, cbar_kws={"label": "FC (Foxf2iKO vs Ctrl)"})
plt.title("neuron probability, values: FDR")
plt.savefig(base / "plots" / "heatmap_ferroptosis_neuron-prob-high-vs-low.svg", bbox_inches="tight")
plt.show()

Exploratory: Foxf2iKO vs Ctrl log2FC of mean expression in 3 equal-width bins of neuron probability

In [ ]:
bins = pd.cut(adata.obs["neuron_prob"], bins=3, labels=["<=0.33", "<=0.67", "<=1"], include_lowest=True)
means = {c: pd.DataFrame({b: np.expm1(np.asarray(adata[(adata.obs["genotype"] == c) & (bins == b), genes].X.mean(0)).ravel()) for b in bins.cat.categories}, index=genes)
         for c in ("Foxf2iKO", "Ctrl")}
plt.figure(figsize=(4, 5))
sns.heatmap(np.log2((means["Foxf2iKO"] + 1e-9) / (means["Ctrl"] + 1e-9)), cmap="coolwarm", center=0, annot=True, fmt=".2f", cbar_kws={"label": "log2FC (Foxf2iKO vs Ctrl)"})
plt.xlabel("neuron probability")
plt.show()

Exploratory: spot-wise expression vs neuron probability

In [ ]:
for cond in ("Ctrl", "Foxf2iKO"):
    sub = adata[adata.obs["genotype"] == cond]
    fig, axs = plt.subplots(int(np.ceil(len(genes) / 4)), 4, figsize=(16, 3.5 * np.ceil(len(genes) / 4)))
    for ax, g in zip(axs.flat, genes):
        x, y = sub.obs["neuron_prob"].values, sub[:, g].X.toarray().ravel()
        sns.scatterplot(x=x, y=y, hue=sub.obs["sample"].values, s=5, alpha=0.6, legend=False, ax=ax)
        sns.regplot(x=x, y=y, scatter=False, color="k", ax=ax)
        ax.set(title=f"{g} (R = {pearsonr(x, y)[0]:.2f})", xlabel="neuron probability", ylabel="expression")
    for ax in axs.flat[len(genes):]:
        ax.set_visible(False)
    fig.suptitle(cond)
    fig.tight_layout()
    plt.show()